# 🔄 Transformation de Données z/OS - Format Pivot

Ce notebook transforme vos données z/OS du format pivot (dates en colonnes) vers le format long requis pour l'entraînement.

**Format Source (vos données):**
```
code_application | code_indicateur | 2022-04-01 | 2022-05-01 | ...
DEV-CICS         | MDIU           | 0,2717     | 0,196      | ...
DEV-CICS         | MPTE           | 0,3014     | 0,3047     | ...
```

**Format Cible (pour ML):**
```
application | timestamp  | M24H | MDIU | MPTE | TXDIU | EFF | TVDIU | MIPS_consumption
DEV-CICS    | 2022-04-01 | 1.23 | 0.27 | 0.30 | 4.92  | ... | ...   | 1.23
```

## 📤 Step 1: Upload Your Pivot Data

Upload your CSV/Excel file with the pivot format.

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import io

print("="*70)
print("📤 UPLOAD YOUR z/OS PIVOT DATA")
print("="*70)
print("\n📋 Expected format:")
print("   - Columns: code_application, code_indicateur, dates...")
print("   - One row per application-indicator combination")
print("   - Dates as column headers (YYYY-MM-DD)")
print("   - Decimal separator: comma (,) or period (.)")
print("\n📊 File types: CSV, TSV, or Excel")
print("\n" + "="*70 + "\n")

# Upload file
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f"\n✅ File uploaded: {filename}")
    print(f"   Size: {len(uploaded[filename]) / 1024:.2f} KB")
    
    SOURCE_FILE = filename
else:
    print("\n⚠️  No file uploaded")
    SOURCE_FILE = None

## 🔍 Step 2: Load and Inspect Data

In [ ]:
if SOURCE_FILE:
    print("🔍 Loading data...\n")
    
    # Try different separators and encodings
    def try_load_file(filepath):
        # Try Excel first
        if filepath.endswith(('.xlsx', '.xls')):
            try:
                return pd.read_excel(filepath)
            except:
                pass
        
        # Try CSV with different separators
        for sep in ['\t', ',', ';']:
            for encoding in ['utf-8', 'latin1', 'iso-8859-1']:
                try:
                    df = pd.read_csv(filepath, sep=sep, encoding=encoding)
                    if len(df.columns) > 2:  # Must have at least 3 columns
                        print(f"✅ Loaded with separator='{sep}', encoding='{encoding}'")
                        return df
                except:
                    continue
        
        raise ValueError("Could not load file. Try saving as CSV UTF-8 or Excel.")
    
    # Load
    df_raw = try_load_file(SOURCE_FILE)
    
    print(f"\n📊 Data loaded:")
    print(f"   Rows: {len(df_raw)}")
    print(f"   Columns: {len(df_raw.columns)}")
    print(f"\n📋 Column names:")
    print(f"   {list(df_raw.columns[:10])}...")
    
    print("\n🔍 First few rows:")
    display(df_raw.head())
    
else:
    print("⚠️  Please upload a file first")

## 🔄 Step 3: Transform to Long Format

This will:
1. Fix decimal separators (comma → period)
2. Unpivot dates to rows
3. Pivot indicators to columns
4. Create MIPS_consumption target variable

In [ ]:
def transform_pivot_to_long(df_raw):
    """
    Transform pivot format to long format for ML.
    """
    print("🔄 Transforming data...\n")
    
    # Step 1: Identify columns
    app_col = df_raw.columns[0]  # First column is application
    ind_col = df_raw.columns[1]  # Second column is indicator
    date_cols = df_raw.columns[2:]  # Rest are dates
    
    print(f"✅ Identified columns:")
    print(f"   Application column: {app_col}")
    print(f"   Indicator column: {ind_col}")
    print(f"   Date columns: {len(date_cols)} dates from {date_cols[0]} to {date_cols[-1]}")
    
    # Step 2: Melt from wide to long
    print("\n📊 Step 1: Unpivoting dates...")
    df_melted = df_raw.melt(
        id_vars=[app_col, ind_col],
        value_vars=date_cols,
        var_name='timestamp',
        value_name='value'
    )
    print(f"   Rows after melt: {len(df_melted):,}")
    
    # Step 3: Fix decimal separator (comma to period)
    print("\n🔧 Step 2: Fixing decimal separators...")
    df_melted['value'] = df_melted['value'].astype(str).str.replace(',', '.')
    df_melted['value'] = pd.to_numeric(df_melted['value'], errors='coerce')
    print(f"   Converted to numeric")
    
    # Step 4: Pivot indicators to columns
    print("\n📊 Step 3: Pivoting indicators to columns...")
    df_long = df_melted.pivot_table(
        index=[app_col, 'timestamp'],
        columns=ind_col,
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Rename application column
    df_long.rename(columns={app_col: 'application'}, inplace=True)
    
    # Remove multi-index from columns
    df_long.columns.name = None
    
    print(f"   Final shape: {df_long.shape}")
    print(f"   Columns: {list(df_long.columns)}")
    
    # Step 5: Create MIPS_consumption target
    print("\n🎯 Step 4: Creating target variable...")
    
    # Option 1: Use M24H as target (most common MIPS measure)
    if 'M24H' in df_long.columns:
        df_long['MIPS_consumption'] = df_long['M24H']
        print(f"   ✅ Using M24H as MIPS_consumption")
    # Option 2: Calculate average of available MIPS indicators
    elif 'MDIU' in df_long.columns and 'MPTE' in df_long.columns:
        df_long['MIPS_consumption'] = (df_long['MDIU'] + df_long['MPTE']) / 2
        print(f"   ✅ Calculated MIPS_consumption as average of MDIU and MPTE")
    else:
        print(f"   ⚠️  Could not determine MIPS_consumption - using first numeric column")
        numeric_cols = df_long.select_dtypes(include=[np.number]).columns
        df_long['MIPS_consumption'] = df_long[numeric_cols[0]]
    
    # Step 6: Ensure required columns exist
    print("\n✅ Step 5: Validating columns...")
    required_indicators = ['M24H', 'MDIU', 'MPTE', 'TXDIU', 'EFF', 'TVDIU']
    
    for col in required_indicators:
        if col not in df_long.columns:
            print(f"   ⚠️  Missing {col} - creating with zeros")
            df_long[col] = 0.0
        else:
            print(f"   ✅ {col} present")
    
    # Step 7: Reorder columns
    final_columns = ['application', 'timestamp', 'M24H', 'MDIU', 'MPTE', 
                    'TXDIU', 'EFF', 'TVDIU', 'MIPS_consumption']
    
    # Keep only required columns (and any extra ones)
    available_cols = [c for c in final_columns if c in df_long.columns]
    df_long = df_long[available_cols]
    
    # Step 8: Remove rows with missing target
    print("\n🧹 Step 6: Cleaning data...")
    before = len(df_long)
    df_long = df_long.dropna(subset=['MIPS_consumption'])
    after = len(df_long)
    print(f"   Removed {before - after} rows with missing target")
    
    # Remove rows where all indicators are zero
    indicator_cols = ['M24H', 'MDIU', 'MPTE', 'TXDIU']
    available_indicators = [c for c in indicator_cols if c in df_long.columns]
    
    if available_indicators:
        before = len(df_long)
        df_long = df_long[df_long[available_indicators].sum(axis=1) > 0]
        after = len(df_long)
        print(f"   Removed {before - after} rows with all zeros")
    
    print(f"\n✅ Transformation complete!")
    print(f"   Final dataset: {len(df_long):,} rows × {len(df_long.columns)} columns")
    print(f"   Applications: {df_long['application'].nunique()}")
    print(f"   Date range: {df_long['timestamp'].min()} to {df_long['timestamp'].max()}")
    
    return df_long

# Transform
if SOURCE_FILE:
    df_transformed = transform_pivot_to_long(df_raw)
else:
    print("⚠️  No data to transform")

## 📊 Step 4: Preview Transformed Data

In [ ]:
if 'df_transformed' in locals():
    print("📊 Transformed Data Preview:\n")
    print("="*70)
    
    # Show first rows
    print("\n🔍 First 10 rows:")
    display(df_transformed.head(10))
    
    # Statistics
    print("\n📈 Statistics:")
    display(df_transformed.describe())
    
    # Data quality
    print("\n✅ Data Quality Check:")
    print(f"   Total records: {len(df_transformed):,}")
    print(f"   Unique applications: {df_transformed['application'].nunique()}")
    print(f"   Date range: {df_transformed['timestamp'].min()} to {df_transformed['timestamp'].max()}")
    print(f"   MIPS range: {df_transformed['MIPS_consumption'].min():.4f} to {df_transformed['MIPS_consumption'].max():.4f}")
    print(f"   Missing values: {df_transformed.isnull().sum().sum()}")
    
    # Applications
    print(f"\n📋 Applications ({df_transformed['application'].nunique()}):")
    print(f"   {df_transformed['application'].unique()[:20]}...")
    
else:
    print("⚠️  No transformed data available")

## 💾 Step 5: Save Transformed Data

In [ ]:
if 'df_transformed' in locals():
    # Save to CSV
    output_filename = 'zos_mips_transformed.csv'
    df_transformed.to_csv(output_filename, index=False)
    
    print(f"✅ Transformed data saved to: {output_filename}")
    print(f"   Size: {len(open(output_filename).read()) / 1024:.2f} KB")
    print(f"   Format: CSV, UTF-8 encoding")
    
    # Set for training
    DATA_PATH = output_filename
    print(f"\n✅ DATA_PATH set to: {DATA_PATH}")
    print("\n🚀 Ready for training!")
    
    # Download option
    print("\n📥 Download transformed file:")
    files.download(output_filename)
    
else:
    print("⚠️  No data to save")

## 🎯 Step 6: Use for Training

Now you can use the transformed data for training!

In [ ]:
if 'DATA_PATH' in locals():
    print("="*70)
    print("🎯 READY FOR TRAINING")
    print("="*70)
    
    print(f"\nTransformed file: {DATA_PATH}")
    print(f"Records: {len(df_transformed):,}")
    print(f"Format: ML-ready (long format)")
    
    print("\n📋 Next steps:")
    print("\n   Option A: Use main notebook")
    print("   1. Go to 03_colab_full_pipeline.ipynb")
    print("   2. Set USE_UPLOADED_DATA = True")
    print("   3. Upload this transformed file")
    print("   4. Continue training")
    
    print("\n   Option B: Train immediately (run cell below)")
    
    print("\n" + "="*70)
else:
    print("⚠️  Transform data first")

## 🚀 BONUS: Train Immediately (Optional)

Uncomment and run this cell to train models immediately with your transformed data.

In [ ]:
# Uncomment to train immediately

# # Setup
# import sys
# import os
#
# # Clone repo if not already done
# if not os.path.exists('Perf_Plan'):
#     !git clone https://github.com/chelvy/Perf_Plan.git
#
# os.chdir('Perf_Plan')
# !pip install -q -r requirements.txt
#
# # Copy transformed data to Perf_Plan directory
# import shutil
# shutil.copy(f'../{DATA_PATH}', 'data/')
#
# # Train
# !python main.py train \
#     --data-path data/{DATA_PATH} \
#     --mode both \
#     --feature-engineering \
#     --test-size 0.2
#
# print("\n✅ Training complete! Check models/ and results/ directories")

---

## 📚 Summary

This notebook:
1. ✅ Loaded your pivot-format z/OS data
2. ✅ Fixed decimal separators (comma → period)
3. ✅ Unpivoted dates from columns to rows
4. ✅ Pivoted indicators from rows to columns
5. ✅ Created MIPS_consumption target variable
6. ✅ Cleaned and validated the data
7. ✅ Saved in ML-ready format

**Your data is now ready for machine learning training! 🎉**

Next: Use the transformed CSV file in the main training notebook.